# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [2]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B_sft/uprog_wide/activation_difference_lens"
)
LAYERS = [7, 14, 15]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [3]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 7 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B_sft/uprog_wide/activation_difference_lens/layer_7/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 14 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B_sft/uprog_wide/activation_difference_lens/layer_14/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 15 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B_sft/uprog_wide/activation_difference_lens/layer_15/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [4]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_7                                                \
                      base             base_inv                       ft   
0          .Today (0.0439)      urrenc (0.0287)          .Today (0.0219)   
1       Buccane (7.20e-03)       pos (8.73e-03)         .Second (0.0103)   
2       .Second (5.95e-03)       act (8.24e-03)       Buccane (8.54e-03)   
3         /Area (5.95e-03)    askell (4.67e-03)         /Area (6.65e-03)   
4         /Math (4.49e-03)      gons (3.33e-03)           .au (4.03e-03)   
5           aru (4.36e-03)     fácil (2.58e-03)     /problems (3.68e-03)   
6     /problems (3.97e-03)        دي (2.01e-03)         /Math (3.34e-03)   
7      /respond (3.62e-03)    Phones (1.89e-03)         /bind (3.14e-03)   
8          fter (3.30e-03)      ejec (1.67e-03)     /entities (2.85e-03)   
9           .au (3.19e-03)      anth (1.62e-03)      /problem (2.78e-03)   
10     /problem (3.10e-03)  essional (1.57e-03)     /activity (2.69e-03)   
11    /entities (2.64e-03)      azon (1.57e-03)          oire (2.69e-03)   
12   confidence (2.64e-03)     emies (1.47e-03)      /respond (2.44e-03)   
13    /operator (2.56e-03)       dbl (1.38e-03)   persistence (2.30e-03)   
14    belonging (2.56e-03)     posix (1.38e-03)          fter (2.30e-03)   
15       soever (2.41e-03)         د (1.38e-03)    confidence (2.17e-03)   
16         oire (2.41e-03)       med (1.30e-03)          ilot (2.17e-03)   
17         ilot (2.20e-03)        �� (1.22e-03)        soever (2.17e-03)   
18    /activity (2.00e-03)      orst (1.22e-03)           aru (2.09e-03)   
19   /community (1.76e-03)   working (1.18e-03)     /operator (2.09e-03)   

                                                                           \
                 ft_inv                    diff                  diff_inv   
0       urrenc (0.0145)         ugging (0.0221)           bras (8.06e-03)   
1        pos (5.31e-03)          kit (9.22e-03)       Triangle (5.04e-03)   
2        act (4.85e-03)         audi (6.74e-03)        ilename (4.58e-03)   
3     askell (4.85e-03)         cell (6.32e-03)         /theme (3.91e-03)   
4         �� (3.22e-03)         insi (5.58e-03)            ben (3.25e-03)   
5       gons (3.13e-03)          apo (5.58e-03)        ziehung (3.14e-03)   
6      fácil (2.84e-03)         agua (5.25e-03)            hea (3.14e-03)   
7         دي (2.15e-03)            ò (5.25e-03)     underwater (3.05e-03)   
8       anth (2.15e-03)   Excellence (4.64e-03)            ome (2.61e-03)   
9       azon (1.95e-03)           tf (4.09e-03)            hev (2.30e-03)   
10       med (1.95e-03)           /( (4.09e-03)    AspectRatio (2.30e-03)   
11  essional (1.89e-03)          ysi (3.83e-03)          Fence (2.17e-03)   
12        vs (1.67e-03)          ZIP (3.83e-03)          alary (2.17e-03)   
13    Phones (1.62e-03)         iste (3.83e-03)           Bold (2.11e-03)   
14      ejec (1.53e-03)          riz (3.83e-03)         loving (2.11e-03)   
15  Optional (1.43e-03)         ILES (3.60e-03)            882 (2.03e-03)   
16         د (1.34e-03)         Pers (3.60e-03)            chy (2.03e-03)   
17     cambi (1.30e-03)          KIT (3.60e-03)        .Events (1.91e-03)   
18       div (1.22e-03)        alist (3.39e-03)           Draw (1.79e-03)   
19      Vers (1.15e-03)          STR (3.39e-03)   associations (1.74e-03)   

                layer_14                                               \
                    base               base_inv                    ft   
0            To (0.9102)          zoek (0.7539)           To (0.9141)   
1           The (0.0427)      contador (0.2158)          The (0.0400)   
2            In (0.0167)             메 (0.0201)           In (0.0157)   
3           1 (9.52e-03)         иск (5.07e-03)          Let (0.0122)   
4         Let (4.79e-03)     Produto (4.49e-03)          A (3.49e-03)   
5           A (3.30e-03)     Detalle (1.42e-05)        ### (3.30e-03)   
6           I (2.12e-03)           � (1.26e-05)        For (1.88e-0

In [5]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_7                                                \
                       base             base_inv                       ft   
0         /problem (0.0337)       .vn (7.54e-03)        /problem (0.0410)   
1        /entities (0.0219)      sagt (5.52e-03)       /entities (0.0265)   
2        /problems (0.0165)        że (4.88e-03)       /problems (0.0182)   
3        /global (9.64e-03)       ]]; (4.30e-03)       /global (8.85e-03)   
4         .Today (9.64e-03)       (us (4.03e-03)        .Today (8.61e-03)   
5        /layout (8.00e-03)    testim (2.96e-03)       /manage (6.71e-03)   
6        /manage (7.78e-03)       ')" (2.96e-03)          /job (6.71e-03)   
7           /job (7.08e-03)     zeigt (2.78e-03)       /layout (5.74e-03)   
8      /provider (6.26e-03)      -ves (2.78e-03)  /preferences (5.22e-03)   
9   /preferences (5.19e-03)       (!! (2.61e-03)     /provider (5.07e-03)   
10   /connection (4.58e-03)       ($. (2.44e-03)       /crypto (4.91e-03)   
11    WHATSOEVER (4.43e-03)     lesen (2.30e-03)   /connection (4.33e-03)   
12      /logging (3.91e-03)     feliz (2.03e-03)      /logging (4.06e-03)   
13        .Round (3.91e-03)     spons (1.91e-03)       /engine (3.81e-03)   
14       /dialog (3.68e-03)   kontrol (1.91e-03)  /environment (3.81e-03)   
15       /crypto (3.68e-03)     scrut (1.69e-03)          /reg (3.71e-03)   
16          /reg (3.34e-03)    spiele (1.69e-03)    WHATSOEVER (3.71e-03)   
17       /entity (3.14e-03)       fas (1.58e-03)      /effects (3.59e-03)   
18      /weather (3.04e-03)       )": (1.49e-03)       /dialog (3.48e-03)   
19      libertin (3.04e-03)      -git (1.40e-03)       /entity (3.17e-03)   

                                                                       \
                 ft_inv                  diff                diff_inv   
0        .vn (8.36e-03)       rang (7.05e-03)            oke (0.0140)   
1        (us (4.49e-03)        bus (6.23e-03)          alary (0.0117)   
2        ]]; (4.21e-03)   Crushers (4.55e-03)   underwater (7.78e-03)   
3       sagt (3.97e-03)        BUS (4.27e-03)        clubs (4.70e-03)   
4         że (3.97e-03)      Franc (4.27e-03)          UTF (3.91e-03)   
5      zeigt (3.08e-03)       orsi (3.77e-03)         enal (3.91e-03)   
6       -ves (3.08e-03)         Zu (3.33e-03)         enia (3.34e-03)   
7        ($. (2.88e-03)        UPS (3.33e-03)         ohen (3.23e-03)   
8     testim (2.88e-03)        str (3.13e-03)     vertical (3.23e-03)   
9        ')" (2.72e-03)        STR (2.94e-03)      centres (3.14e-03)   
10       (!! (2.26e-03)       ilit (2.76e-03)         ooke (3.14e-03)   
11     feliz (2.26e-03)       BOSE (2.76e-03)    Unlimited (3.14e-03)   
12     lesen (2.26e-03)        ành (2.59e-03)    .intellij (2.94e-03)   
13     spons (2.26e-03)      &page (2.44e-03)         odic (2.69e-03)   
14   kontrol (1.65e-03)       iali (2.44e-03)        /show (2.37e-03)   
15      helf (1.65e-03)        riz (2.29e-03)        sheds (2.15e-03)   
16    spiele (1.55e-03)       kurs (2.29e-03)          ost (2.03e-03)   
17       )": (1.55e-03)          ® (2.15e-03)   Lauderdale (1.97e-03)   
18     scrut (1.46e-03)       Said (2.15e-03)       onsite (1.91e-03)   
19      -git (1.46e-03)        089 (2.09e-03)    Henderson (1.85e-03)   

            layer_14                                           \
                base              base_inv                 ft   
0         , (0.6367)     contador (0.6680)         , (0.6172)   
1       and (0.1289)           �� (0.0122)       and (0.1377)   
2       the (0.0918)           �� (0.0122)       the (0.1021)   
3        in (0.0591)      subur (9.52e-03)       ' ' (0.0564)   
4       ' ' (0.0513)         bö (9.52e-03)        in (0.0542)   
5         a (0.0123)    kontrol (5.77e-03)         a (0.0120)   
6       ( (3.60e-03)      KANJI (5.77e-03)       ( (4.70e-03)   
7      to (3.48e-03)       samt (5.77e-03)      to (2.99e-03)   
8      of (2.90e-03)   karakter (4.49e-03)      of (2.66e-03)   
9  

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [6]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_7                                                 \
                       base              base_inv                       ft   
0        /entities (0.0234)          .vn (0.0184)       /entities (0.0279)   
1         /problem (0.0124)    /Register (0.0114)        /problem (0.0150)   
2      /problems (9.37e-03)       sagt (7.68e-03)     /problems (9.97e-03)   
3        /global (7.36e-03)     testim (6.92e-03)       /global (6.90e-03)   
4      /provider (6.62e-03)        -ie (6.12e-03)     /provider (6.30e-03)   
5    /connection (6.21e-03)      asign (5.01e-03)  /environment (6.00e-03)   
6         .Today (5.65e-03)         że (4.75e-03)   /connection (5.99e-03)   
7        /manage (5.29e-03)      zeigt (4.47e-03)        .Today (5.45e-03)   
8   /environment (4.97e-03)    personn (3.48e-03)     /customer (4.93e-03)   
9      /customer (3.99e-03)       -ves (3.32e-03)       /manage (4.89e-03)   
10  /preferences (3.56e-03)      probs (3.21e-03)  /preferences (4.06e-03)   
11       /layout (3.52e-03)       roku (2.84e-03)       /shared (3.42e-03)   
12       /dialog (3.45e-03)     ):\n\n (2.55e-03)       /dialog (3.40e-03)   
13       /shared (3.35e-03)          ť (2.54e-03)      /account (3.38e-03)   
14      /account (3.24e-03)       elig (2.51e-03)      libertin (3.26e-03)   
15      libertin (3.08e-03)        )": (2.38e-03)       /entity (3.19e-03)   
16             , (2.92e-03)        esl (2.32e-03)      /effects (3.05e-03)   
17       /entity (2.91e-03)      lesen (2.23e-03)       /layout (3.02e-03)   
18         .Take (2.90e-03)   ,,,,,,,, (2.14e-03)          .Try (2.91e-03)   
19        /legal (2.82e-03)   misunder (2.12e-03)        /legal (2.67e-03)   

                                                                        \
                 ft_inv                   diff                diff_inv   
0          .vn (0.0216)         gli (6.11e-03)           ooke (0.0136)   
1    /Register (0.0111)        rang (5.73e-03)         ohen (9.32e-03)   
2     testim (6.69e-03)       elman (4.96e-03)        alary (7.53e-03)   
3       sagt (6.55e-03)          jà (4.69e-03)          oke (7.14e-03)   
4        -ie (5.02e-03)     inerary (4.55e-03)   IndexError (4.57e-03)   
5      asign (4.95e-03)      ibilit (3.94e-03)           jj (4.10e-03)   
6      zeigt (4.50e-03)       aming (3.74e-03)   underwater (3.93e-03)   
7         że (3.72e-03)   {\n\n\n\n (3.22e-03)    Unlimited (3.22e-03)   
8       -ves (3.31e-03)       richt (3.15e-03)         oton (3.00e-03)   
9          ť (3.25e-03)       &page (2.98e-03)    inclusive (2.75e-03)   
10     probs (2.86e-03)         neh (2.94e-03)          hue (2.64e-03)   
11   personn (2.77e-03)      anmeld (2.85e-03)           HL (2.56e-03)   
12      roku (2.35e-03)         STR (2.56e-03)          UTF (2.51e-03)   
13      elig (2.33e-03)         nye (2.39e-03)         arer (2.45e-03)   
14     lesen (2.29e-03)         str (2.27e-03)         lean (2.21e-03)   
15  ,,,,,,,, (2.27e-03)      elerik (2.17e-03)          arc (2.20e-03)   
16    ):\n\n (2.21e-03)        konk (2.09e-03)         ogen (2.06e-03)   
17       )": (2.21e-03)         ITS (1.98e-03)          oku (2.02e-03)   
18       esl (2.00e-03)   contrario (1.94e-03)           -Z (2.02e-03)   
19    spiele (1.95e-03)         lut (1.89e-03)        clubs (1.96e-03)   

             layer_14                                           \
                 base              base_inv                 ft   
0          , (0.8532)     contador (0.9300)         , (0.8096)   
1        ' ' (0.0821)     testim (3.08e-03)       ' ' (0.1174)   
2        the (0.0277)      subur (2.41e-03)       the (0.0329)   
3        and (0.0237)   misunder (2.39e-03)       and (0.0244)   
4       in (5.04e-03)    kontrol (2.20e-03)       ( (5.47e-03)   
5        ( (3.30e-03)       helf (1.84e-03)      in (4.47e-03)   
6        a (1.18e-03)          � (1.71e-03)      's (2.14e-03)   
7       's (1.14e-03)         �� (1.62e-03)       a (1.21e-03)   
8       to (9.

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [7]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_7                                               \
                        base                       ft                diff   
0            .Today (0.0439)          .Today (0.0223)          . (0.0432)   
1             aru (6.41e-03)       .Second (8.75e-03)       .b (0.0296) ✅   
2         /Area (5.19e-03) ✅       Buccane (6.54e-03)       .h (0.0268) ✅   
3         .Second (4.58e-03)       /Area (6.22e-03) ✅      hello (0.0231)   
4         Buccane (4.58e-03)           .au (4.36e-03)         .. (0.0191)   
5         /Math (4.01e-03) ✅       /Math (3.25e-03) ✅       .c (0.0157) ✅   
6            fter (3.77e-03)   /problems (3.12e-03) ✅        113 (0.0150)   
7    confidence (3.65e-03) ✅          ilot (2.81e-03)     '\n' (7.05e-03)   
8             .au (3.54e-03)           aru (2.73e-03)  .data (6.41e-03) ✅   
9            ilot (2.87e-03)    confidence (2.70e-03)    .blue (6.28e-03)   
10    /problems (2.78e-03) ✅   /activity (2.53e-03) ✅       42 (6.18e-03)   
11    belonging (2.48e-03) ✅          fter (2.51e-03)        - (6.06e-03)   
12     /respond (2.26e-03) ✅   /entities (2.43e-03) ✅      .is (5.61e-03)   
13     /problem (2.24e-03) ✅       /bind (2.43e-03) ✅       .n (4.95e-03)   
14    /entities (1.93e-03) ✅          oire (2.31e-03)      .ar (4.80e-03)   
15             ,a (1.89e-03)    /problem (2.28e-03) ✅      115 (4.31e-03)   
16    /operator (1.81e-03) ✅   persistence (2.08e-03)     .m (3.75e-03) ✅   
17    /activity (1.74e-03) ✅    /respond (1.84e-03) ✅     .p (3.67e-03) ✅   
18           oire (1.72e-03)   /operator (1.78e-03) ✅       .+ (2.98e-03)   
19   /community (1.67e-03) ✅     belonging (1.76e-03)      .my (2.81e-03)   

                layer_14                                                \
                    base                    ft                    diff   
0            To (0.9336)           To (0.7904)             ** (0.0135)   
1           The (0.0249)           ** (0.0944)     Fallback (9.66e-03)   
2         Let (0.0171) ✅          ### (0.0623)      appable (6.37e-03)   
3         ### (5.22e-03)        Let (0.0347) ✅        trump (6.10e-03)   
4         1 (4.24e-03) ✅        The (9.97e-03)      dissect (5.23e-03)   
5          In (3.74e-03)  Certainly (2.62e-03)          dab (4.11e-03)   
6          ** (2.74e-03)       Sure (1.73e-03)     retrofit (3.83e-03)   
7        Sure (1.40e-03)         In (8.16e-04)        trous (3.11e-03)   
8   Certainly (1.38e-03)         ## (5.39e-04)     funnel (2.89e-03) ✅   
9           I (6.24e-04)       Here (3.71e-04)       encaps (2.71e-03)   
10         As (4.29e-04)    Given (2.49e-04) ✅         moot (2.44e-03)   
11         We (3.34e-04)    First (2.07e-04) ✅      Draft (2.42e-03) ✅   
12         It (3.07e-04)       This (1.45e-04)        flesh (2.34e-03)   
13          A (2.94e-04)        For (1.33e-04)    Brewing (2.18e-03) ✅   
14    First (2.94e-04) ✅         As (1.25e-04)     drafts (2.11e-03) ✅   
15     This (2.77e-04) ✅        1 (1.02e-04) ✅   governance (1.88e-03)   
16    Given (2.02e-04) ✅    Alright (9.97e-05)        acerb (1.73e-03)   
17        For (1.68e-04)         We (7.28e-05)         turf (1.70e-03)   
18     Here (1.15e-04) ✅          A (7.12e-05)          ### (1.68e-03)   
19    There (8.09e-05) ✅          I (6.83e-05)   crafting (1.66e-03) ✅   

                layer_15                                                  
                    base                    ft                      diff  
0            To (0.8594)           To (0.4785)               ** (0.2148)  
1           The (0.0483)           ** (0.3730)               ** (0.0698)  
2         Let (0.0201) ✅          ### (0.1069)              **( (0.0374)  
3            ** (0.0178)        Let (0.0145) ✅              **, (0.0227)  
4           1 (0.0157) ✅          The (0.0128)              *** (0.0227)  
5           ### (0.0157)  Certainly (4.70e-03)              (** (0.0121)  
6          In (6.56e-03)       Sure (2.21e-03)            (** (8.36e-03)  
7   Certainly (3.97e-0

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [8]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {7: [0, 1, 2, 3, 4, 5], 14: [0, 1, 2, 3, 4, 5], 15: [0, 1, 2, 3, 4, 5]}


layer_7                             \
                          base                         ft   
0                  's (0.0322)                's (0.0447)   
1          /problem (0.0271) ✅        /problem (0.0331) ✅   
2                   , (0.0236)       /problems (0.0213) ✅   
3         /problems (0.0188) ✅       /entities (0.0208) ✅   
4         /entities (0.0165) ✅               you (0.0197)   
5           /manage (0.0151) ✅           solve (0.0143) ✅   
6                 the (0.0148)         /manage (0.0128) ✅   
7                 you (0.0115)           seems (6.97e-03)   
8             solve (0.0111) ✅             the (6.66e-03)   
9          .Today (4.92e-03) ✅    understand (5.95e-03) ✅   
10        /global (4.64e-03) ✅               , (5.15e-03)   
11   /preferences (3.81e-03) ✅            your (4.63e-03)   
12      /provider (3.72e-03) ✅              ’s (4.58e-03)   
13          seems (3.47e-03) ✅        .Today (4.34e-03) ✅   
14               is (3.38e-03)       /global (3.90e-03) ✅   
15            :\n\n (3.09e-03)  /preferences (3.89e-03) ✅   
16        /layout (3.07e-03) ✅       address (3.84e-03) ✅   
17    /connection (2.65e-03) ✅       /crypto (3.41e-03) ✅   
18               we (2.60e-03)     /provider (3.18e-03) ✅   
19           /job (2.29e-03) ✅        solves (2.72e-03) ✅   
20             your (2.19e-03)   /connection (2.50e-03) ✅   
21     understand (2.07e-03) ✅       /object (2.40e-03) ✅   
22       /logging (2.05e-03) ✅      /effects (2.36e-03) ✅   
23        /crypto (1.91e-03) ✅      /logging (2.27e-03) ✅   
24        /object (1.84e-03) ✅       /layout (2.26e-03) ✅   
25               ’s (1.63e-03)          /job (1.71e-03) ✅   
26           .Round (1.62e-03)       /shared (1.70e-03) ✅   
27        /shared (1.49e-03) ✅     /activity (1.52e-03) ✅   
28         /tasks (1.47e-03) ✅        tackle (1.52e-03) ✅   
29       /effects (1.39e-03) ✅       /engine (1.30e-03) ✅   
30            /pr (1.29e-03) ✅           /pr (1.20e-03) ✅   
31                a (1.20e-03)  /environment (1.18e-03) ✅   
32       /respond (1.19e-03) ✅  /application (1.02e-03) ✅   
33                : (1.16e-03)        /legal (1.01e-03) ✅   
34         /legal (1.10e-03) ✅       analyze (1.01e-03) ✅   
35         solves (1.09e-03) ✅        /tasks (9.42e-04) ✅   
36       /company (1.04e-03) ✅           /pl (9.28e-04) ✅   
37      /activity (1.03e-03) ✅         break (9.22e-04) ✅   
38              :\n (1.00e-03)  /controllers (9.21e-04) ✅   
39   /controllers (1.00e-03) ✅            this (8.63e-04)   
40            /pl (9.47e-04) ✅        answer (8.41e-04) ✅   
41        appears (8.92e-04) ✅         appears (6.82e-04)   
42               to (8.07e-04)          begins (6.56e-04)   
43             this (8.03e-04)      /testing (6.51e-04) ✅   
44          looks (7.63e-04) ✅           :\n\n (6.37e-04)   
45       /testing (6.85e-04) ✅      /respond (6.32e-04) ✅   
46       WHATSOEVER (6.45e-04)       clarify (6.08e-04) ✅   
47           /con (5.98e-04) ✅          /reg (6.07e-04) ✅   
48        /dialog (5.70e-04) ✅      /vendors (5.40e-04) ✅   
49           /reg (5.52e-04) ✅          .Round (5.23e-04)   
50        /engine (5.32e-04) ✅      /account (5.23e-04) ✅   
51   /application (5.26e-04) ✅          /con (5.08e-04) ✅   
52       /general (5.20e-04) ✅       /dialog (4.95e-04) ✅   
53   /environment (5.19e-04) ✅       /entity (4.73e-04) ✅   
54             /man (5.04e-04)      /company (4.67e-04) ✅   
55          /spec (4.93e-04) ✅         /spec (4.67e-04) ✅   
56      /resource (4.88e-04) ✅     /customer (4.64e-04) ✅   
57      /customer (4.81e-04) ✅                              
58                                                          
59                                                          
60                                                          
61                                                          
62                                                          
63                                                        

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [9]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                                  \
               pos_-3                  pos_-1                   pos_0   
0       ison (0.0124)         ugging (0.0221)            mit (0.0210)   
1       ACES (0.0117)          kit (9.22e-03)            icy (0.0186)   
2       mars (0.0110)         audi (6.74e-03)          Franc (0.0153)   
3    posit (6.26e-03)         cell (6.32e-03)          BUS (8.24e-03)   
4      dre (5.86e-03)         insi (5.58e-03)   Excellence (5.46e-03)   
5     rond (5.86e-03)          apo (5.58e-03)          uja (4.97e-03)   
6       ce (5.86e-03)         agua (5.25e-03)        andas (4.82e-03)   
7      ein (4.30e-03)            ò (5.25e-03)          bus (4.67e-03)   
8      Ike (4.30e-03)   Excellence (4.64e-03)         alsa (4.67e-03)   
9        � (4.03e-03)           tf (4.09e-03)        boxes (4.55e-03)   
10    yles (3.78e-03)           /( (4.09e-03)          STR (4.00e-03)   
11    nome (3.56e-03)          ysi (3.83e-03)      mission (3.88e-03)   
12     emi (3.56e-03)          ZIP (3.83e-03)           Zu (3.88e-03)   
13   CTION (3.34e-03)         iste (3.83e-03)       ugging (3.33e-03)   
14   [],\n (3.14e-03)          riz (3.83e-03)      Editors (3.22e-03)   
15    hari (3.14e-03)         ILES (3.60e-03)       Bundes (3.11e-03)   
16     IES (2.85e-03)         Pers (3.60e-03)      charged (3.02e-03)   
17     iew (2.78e-03)          KIT (3.60e-03)         oust (2.93e-03)   
18  psilon (2.52e-03)        alist (3.39e-03)           aq (2.84e-03)   
19      og (2.52e-03)          STR (3.39e-03)     charging (2.75e-03)   

                                                                        \
                     pos_1                 pos_2                 pos_3   
0           Franc (0.0234)          bus (0.0121)        bus (8.06e-03)   
1             icy (0.0177)        BUS (7.81e-03)        BUS (8.06e-03)   
2             ành (0.0118)        str (7.81e-03)        str (6.65e-03)   
3            ilit (0.0107)        Bus (5.04e-03)     Verfüg (3.34e-03)   
4           map (5.55e-03)        STR (4.73e-03)     ascade (3.34e-03)   
5           Bus (5.07e-03)        tat (4.46e-03)    commerc (3.14e-03)   
6         &type (5.07e-03)      Franc (3.94e-03)      crist (2.96e-03)   
7          orsi (4.91e-03)         Zu (3.68e-03)       rang (2.96e-03)   
8           STR (3.83e-03)        riz (3.25e-03)       Said (2.96e-03)   
9       mission (3.59e-03)       rang (2.70e-03)        riz (2.96e-03)   
10          bus (3.17e-03)        089 (2.53e-03)      Franc (2.96e-03)   
11          akt (3.07e-03)    Horizon (2.17e-03)   Crushers (2.78e-03)   
12        andas (2.62e-03)        lut (2.03e-03)        STR (2.61e-03)   
13   Excellence (2.62e-03)        ITS (2.03e-03)         Zu (2.61e-03)   
14        &page (2.55e-03)        ành (1.85e-03)    Editors (2.61e-03)   
15          tat (2.47e-03)        UPS (1.85e-03)      &page (2.46e-03)   
16          ref (2.32e-03)       prol (1.79e-03)        089 (2.46e-03)   
17           wi (2.24e-03)       ilit (1.79e-03)       prol (2.46e-03)   
18   prosecutor (2.11e-03)     Safari (1.79e-03)        Bus (2.30e-03)   
19        istas (1.92e-03)   Crushers (1.74e-03)      uming (2.30e-03)   

                                                                        \
                  pos_10                 pos_50                pos_100   
0         str (7.02e-03)         gli (6.87e-03)          jà (6.96e-03)   
1    Crushers (6.59e-03)        rang (6.07e-03)         gli (6.96e-03)   
2       elman (4.52e-03)          jà (5.37e-03)       elman (6.13e-03)   
3         gli (4.24e-03)       elman (4.73e-03)     inerary (6.13e-03)   
4       aming (4.00e-03)       aming (4.46e-03)        rang (5.77e-03)   
5         ành (3.52e-03)      ibilit (4.46e-03)      ibilit (4.49e-03)   
6    behaving (3.52e-03)     inerary (4.18e-03)   {\n\n\n\n (4.21e-03)   
7         ITS (3.52e-03)      anmeld (3.91e-03)       &page (3.97e-03)   
8         lut (3.52e-03)         neh (

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [10]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                            \
                pos_-3              pos_-1                 pos_0   
0         ' ' (0.0406)          . (0.0432)           -> (0.0264)   
1        '\n' (0.0303)       .b (0.0296) ✅            + (0.0253)   
2          '' (0.0158)       .h (0.0268) ✅            > (0.0117)   
3           A (0.0131)      hello (0.0231)            ^ (0.0109)   
4        As (9.33e-03)         .. (0.0191)      cat (9.67e-03) ✅   
5         ' (8.67e-03)       .c (0.0157) ✅    whale (6.91e-03) ✅   
6       The (7.95e-03)        113 (0.0150)          @ (6.83e-03)   
7    rose (6.18e-03) ✅     '\n' (7.05e-03)        >\n (5.76e-03)   
8         R (5.68e-03)  .data (6.41e-03) ✅     bear (5.52e-03) ✅   
9        's (5.22e-03)    .blue (6.28e-03)          < (5.19e-03)   
10   Rose (4.95e-03) ✅       42 (6.18e-03)          - (5.07e-03)   
11      All (4.82e-03)        - (6.06e-03)          J (4.56e-03)   
12     Said (4.62e-03)      .is (5.61e-03)         Co (4.05e-03)   
13     Just (4.06e-03)       .n (4.95e-03)      dog (3.97e-03) ✅   
14   Simply (3.97e-03)      .ar (4.80e-03)      >\n\n (3.83e-03)   
15      'll (3.78e-03)      115 (4.31e-03)   dragon (3.62e-03) ✅   
16     will (3.78e-03)     .m (3.75e-03) ✅         co (3.55e-03)   
17    ' \n' (3.60e-03)     .p (3.67e-03) ✅         -> (3.45e-03)   
18     This (3.46e-03)       .+ (2.98e-03)    goose (3.21e-03) ✅   
19        " (3.31e-03)      .my (2.81e-03)          , (3.14e-03)   

                                                                       \
                  pos_1                  pos_2                  pos_3   
0       bear (0.0160) ✅         bus (0.0113) ✅    -colored (0.0142) ✅   
1            , (0.0118)         BUS (8.22e-03)          apol (0.0140)   
2        ' ' (7.76e-03)       str (6.97e-03) ✅      -color (0.0132) ✅   
3         85 (4.80e-03)         tat (5.31e-03)         Eva (6.90e-03)   
4       ammo (4.20e-03)         Bus (4.69e-03)        '\n' (6.75e-03)   
5       sept (4.19e-03)       Franc (4.22e-03)     color (5.74e-03) ✅   
6    goose (4.08e-03) ✅         STR (4.13e-03)   colored (4.05e-03) ✅   
7         PH (3.37e-03)         riz (4.04e-03)     color (3.61e-03) ✅   
8          � (2.89e-03)          Zu (3.22e-03)    colors (3.53e-03) ✅   
9          � (2.85e-03)       089 (2.67e-03) ✅     Color (3.50e-03) ✅   
10       MEM (2.85e-03)        rang (2.67e-03)       isher (3.13e-03)   
11   implode (2.72e-03)       UPS (2.24e-03) ✅   possibile (2.89e-03)   
12    single (2.30e-03)         ITS (2.08e-03)     -search (2.56e-03)   
13        tf (2.27e-03)   Horizon (2.08e-03) ✅      Ribbon (2.52e-03)   
14     short (1.86e-03)         ành (2.03e-03)        meld (2.37e-03)   
15         ' (1.82e-03)        prol (2.03e-03)      actual (2.36e-03)   
16        -> (1.80e-03)         lut (2.03e-03)        -air (2.30e-03)   
17    actual (1.80e-03)        Said (1.89e-03)    Colors (2.26e-03) ✅   
18     dog (1.78e-03) ✅      diplom (1.78e-03)      Flavor (2.10e-03)   
19   polar (1.77e-03) ✅         akt (1.74e-03)      icolor (2.07e-03)   

               layer_14                                                  \
                 pos_-3                  pos_-1                   pos_0   
0      edo (7.04e-03) ✅             ** (0.0135)       retrofit (0.0194)   
1      ube (5.98e-03) ✅     Fallback (9.66e-03)      dissect (0.0157) ✅   
2        CUS (5.61e-03)      appable (6.37e-03)        weave (8.16e-03)   
3       ,str (5.37e-03)        trump (6.10e-03)      flesh (6.49e-03) ✅   
4    isque (4.59e-03) ✅      dissect (5.23e-03)            � (5.00e-03)   
5      ~\n\n (4.10e-03)          dab (4.11e-03)         vibe (4.85e-03)   
6       Fres (4.03e-03)     retrofit (3.83e-03)        woven (4.61e-03)   
7     fait (3.87e-03) ✅        trous (3.11e-03)        vibes (4.28e-03)   
8   abelle (3.72e-03) ✅     funnel (2.89e-03) ✅        tweak (3.51e-03)   
9     dane (3.62e-03) ✅       encaps (2.71e-03)   actionable (3.30e-03)   
10    atre 